In [ ]:
# Install required libraries
!pip install imbalanced-learn openpyxl

# Imports
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [ ]:
# Upload file manually in Colab
from google.colab import files
uploaded = files.upload()

# Load dataset
df = pd.read_csv("Cleaned_BankChurners_Final (1).csv")

df.head()

Saving Cleaned_BankChurners_Final (1).csv to Cleaned_BankChurners_Final (1).csv


,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
0,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061
1,Existing Customer,49,F,5,Graduate,Single,Less Than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105
2,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,1524,3418.0,2.594,1887,20,2.333,0.000
3,Existing Customer,40,F,4,High School,Married,Less Than $40K,Blue,34,3,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760
4,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,5,1,0,4716.0,1524,4716.0,2.175,816,28,2.500,0.000


In [ ]:
print(df.columns)

# Convert churn to numeric (if needed)
if "Attrition_Flag" in df.columns:
    df["Attrition_Flag"] = df["Attrition_Flag"].map({
        "Existing Customer": 0,
        "Attrited Customer": 1
    })

print(df["Attrition_Flag"].value_counts())

Index(['Attrition_Flag', 'Customer_Age', 'Gender', 'Dependent_count',
       'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category',
       'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon',
       'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal',
       'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt',
       'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio'],
      dtype='object')
Attrition_Flag
0    8500
1    1627
Name: count, dtype: int64


In [ ]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
num_cols = num_cols.drop("Attrition_Flag")

def cap_outliers(df, cols):
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        df[col] = df[col].clip(lower, upper)
    return df

df = cap_outliers(df, num_cols)

In [ ]:
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col + "_outlier"] = ((df[col] < lower) | (df[col] > upper)).astype(int)

In [ ]:
df.to_excel("Updated_With_Outliers.xlsx", index=False)

In [ ]:
X = df.drop("Attrition_Flag", axis=1)
y = df["Attrition_Flag"]

# Identify categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns

# Apply one-hot encoding to categorical columns
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

In [ ]:
pipeline = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("model", RandomForestClassifier(random_state=42))
])

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(pipeline, X, y, cv=cv, scoring='f1')

print("F1 Scores:", scores)
print("Mean F1 Score:", scores.mean())

F1 Scores: [0.85935085 0.87349398 0.82704403 0.85895404 0.8516129 ]
Mean F1 Score: 0.8540911591136741


In [ ]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20]
}

grid = GridSearchCV(pipeline, param_grid, cv=cv, scoring='f1', n_jobs=-1)
grid.fit(X, y)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

Best Parameters: {'model__max_depth': None, 'model__n_estimators': 200}
Best Score: 0.8586557763632877


In [ ]:
best_model = grid.best_estimator_

# Fit on full data
best_model.fit(X, y)

y_pred = best_model.predict(X)

print(classification_report(y, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      8500
           1       1.00      1.00      1.00      1627

    accuracy                           1.00     10127
   macro avg       1.00      1.00      1.00     10127
weighted avg       1.00      1.00      1.00     10127



In [ ]:
model = best_model.named_steps["model"]

importances = model.feature_importances_

for name, val in zip(X.columns, importances):
    print(name, val)

Customer_Age 0.029832028286085582
Dependent_count 0.023649885570049437
Months_on_book 0.022712476612243775
Total_Relationship_Count 0.057498890447386757
Months_Inactive_12_mon 0.07041391193890949
Contacts_Count_12_mon 0.05988499210808817
Credit_Limit 0.026415794891445576
Total_Revolving_Bal 0.026776485419411456
Avg_Open_To_Buy 0.029121700828364808
Total_Amt_Chng_Q4_Q1 0.0516854079447495
Total_Trans_Amt 0.17262910323901187
Total_Trans_Ct 0.19470744640304252
Total_Ct_Chng_Q4_Q1 0.10485619789535593
Avg_Utilization_Ratio 0.07600829259023612
Customer_Age_outlier 0.0
Dependent_count_outlier 0.0
Months_on_book_outlier 0.0
Total_Relationship_Count_outlier 0.0
Months_Inactive_12_mon_outlier 0.0
Contacts_Count_12_mon_outlier 0.0
Credit_Limit_outlier 0.0
Total_Revolving_Bal_outlier 0.0
Avg_Open_To_Buy_outlier 0.0
Total_Amt_Chng_Q4_Q1_outlier 0.0
Total_Trans_Amt_outlier 0.0
Total_Trans_Ct_outlier 0.0
Total_Ct_Chng_Q4_Q1_outlier 0.0
Avg_Utilization_Ratio_outlier 0.0
Gender_M 0.011300681461215021
Ed

In [ ]:
df.to_excel("Final_Preprocessed_Data.xlsx", index=False)